# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [190]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [191]:
# Initialization

load_dotenv(override=True)

OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL")
MODEL = os.getenv("OPENROUTER_GPT_OSS")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

openai_api_key = OPENROUTER_API_KEY # os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = MODEL # "gpt-4.1-mini"
openai = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY)

DB = "prices.db"

OpenAI API Key exists and begins sk-or-v1


In [192]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [193]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [194]:
get_ticket_price("Paris")

DATABASE TOOL CALLED: Getting price for Paris


'Ticket price to Paris is $899.0'

In [195]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [196]:

def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7884
* To create a public link, set `share=True` in `launch()`.


In [197]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [198]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [199]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7885
* To create a public link, set `share=True` in `launch()`.


## A bit more about what Gradio actually does:

1. Gradio constructs a frontend Svelte app based on our Python description of the UI
2. Gradio starts a server built upon the Starlette web framework listening on a free port that serves this React app
3. Gradio creates backend routes for our callbacks, like chat(), which calls our functions

And of course when Gradio generates the frontend app, it ensures that the the Submit button calls the right backend route.

That's it!

It's simple, and it has a result that feels magical.

# Let's go multi-modal!!

We can use DALL-E-3, the image generation model behind GPT-4o, to make us some images

Let's put this in a function called artist.

### Price alert: each time I generate an image it costs about 4 cents - don't go crazy with images!

In [200]:
# Some imports for handling images

import base64
from io import BytesIO
from PIL import Image

In [201]:
def artist(city):
    msg = openai.chat.completions.create(
        model="google/gemini-3.1-flash-image-preview", 
        messages=[{"role": "user", "content": f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style"}],
        modalities=["image"]
    ).model_dump()['choices'][0]['message']
    
    url = msg['images'][0]['image_url']['url'] if msg.get('images') else msg['content']
    return Image.open(BytesIO(base64.b64decode(url.split(",")[1])))
    
def artist_fullCode(city):
    # 2. Use chat.completions instead of images.generate
    response = openai.chat.completions.create(
        model="google/gemini-3.1-flash-image-preview", # A valid image model on OpenRouter
        messages=[
            {
                "role": "user",
                "content": f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style"
            }
        ],
        # 3. REQUIRED: This tells OpenRouter you want a picture
        modalities=["image"] 
    )

    # 4. Safely extract OpenRouter's custom response format
    # We use model_dump() because OpenRouter adds an 'images' array 
    # that the standard OpenAI Python SDK might otherwise hide.
    response_dict = response.model_dump()
    message = response_dict['choices'][0]['message']
    
    # OpenRouter puts the base64 Data URL here:
    if 'images' in message and message['images']:
        image_data_url = message['images'][0]['image_url']['url']
    else:
        # Fallback if the provider sticks it in the text content
        image_data_url = message.get('content', '')

    # 5. Decode the Data URL (data:image/png;base64,iVBORw0KGgo...)
    try:
        # Split at the comma to remove the header and keep only the base64 string
        base64_str = image_data_url.split(",")[1]
        image_data = base64.b64decode(base64_str)
        return Image.open(BytesIO(image_data))
    except IndexError:
        print("Failed to parse image. Make sure you have credits on OpenRouter.")
        print(f"Raw response: {image_data_url[:200]}")
        return None
def artist2(city):
    image_response = openai.images.generate(
            model="dall-e-3",
            prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
            size="1024x1024",
            n=1,
            response_format="b64_json",
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [202]:
image = artist("Sonipat, Haryana, India")
display(image)

KeyboardInterrupt: 

In [203]:
import io
import wave
def talker(message):
    response_stream = openai.chat.completions.create(
        model="openai/gpt-4o-audio-preview", 
        messages=[{"role": "user", "content": f"Read this text out loud exactly as written: {message}"}],
        modalities=["text", "audio"], 
        audio={"voice": "onyx", "format": "pcm16"}, 
        stream=True 
    )

    audio_base64 = ""
    
    for chunk in response_stream:
        chunk_dict = chunk.model_dump()
        
        # THE FIX: Safely skip empty chunks (like the start/end signals of the stream)
        if not chunk_dict.get('choices'):
            continue
            
        delta = chunk_dict['choices'][0].get('delta', {})
        
        # Safely extract the audio data if it exists in this chunk
        if 'audio' in delta and delta.get('audio') and 'data' in delta['audio']:
            audio_base64 += delta['audio']['data']

    # Decode the accumulated base64 string back into raw pcm16 bytes
    pcm_bytes = base64.b64decode(audio_base64)
    
    # Package the raw bytes into a proper WAV file
    wav_io = io.BytesIO()
    with wave.open(wav_io, 'wb') as wav_file:
        wav_file.setnchannels(1)      # Mono audio
        wav_file.setsampwidth(2)      # 16-bit audio
        wav_file.setframerate(24000)  # 24,000 Hz sample rate
        wav_file.writeframes(pcm_bytes)
    
    # Save to disk and return the path for Gradio
    output_filepath = "output_voice.wav"
    with open(output_filepath, "wb") as f:
        f.write(wav_io.getvalue())
        
    return output_filepath

In [ ]:
file_path = talker("Hello, How are you?")
print(f"Saved audio to: {file_path}")

Saved audio to: output_voice.wav


## Let's bring this home:

1. A multi-modal AI assistant with image and audio generation
2. Tool callling with database lookup
3. A step towards an Agentic workflow


In [212]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    system_message = "when you have to call the ticket price function, after telling him ticket price also tell user a very short descripton about Tourist sports in the city he want visit, on Tourist point of view only."

    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)

    if cities:
        image = artist(cities[0])
    
    return history, voice, image


In [213]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses, cities

## The 3 types of Gradio UI

`gr.Interface` is for standard, simple UIs

`gr.ChatInterface` is for standard ChatBot UIs

`gr.Blocks` is for custom UIs where you control the components and the callbacks

In [217]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(share=True)

* Running on local URL:  http://127.0.0.1:7891
* Running on public URL: https://4dc119b99a77c8e796.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


DATABASE TOOL CALLED: Getting price for Gohana


# Exercises and Business Applications

Add in more tools - perhaps to simulate actually booking a flight. A student has done this and provided their example in the community contributions folder.

Next: take this and apply it to your business. Make a multi-modal AI assistant with tools that could carry out an activity for your work. A customer support assistant? New employee onboarding assistant? So many possibilities! Also, see the week2 end of week Exercise in the separate Notebook.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a HUGE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>